# Model notebook, first version (earlier submission)

This notebook reproduces our **earlier submission** (public LB ROC-AUC 0.96594, pooled OOF ROC-AUC 0.9683).
It is kept for reproducibility of both selected submissions. The **final model** is `02_model.ipynb`.

Run on Kaggle with the competition data attached, internet off, CPU only: Run All. Output:
`/kaggle/working/submission.csv`, `oof.csv` (with fold assignments) and `cv_report.json`.

Known validation issues, fixed in `02_model.ipynb` and described in Section 4.3 of the report:
- LightGBM early-stops on the held-out fold (`eval_set` = validation fold), so the number of trees is
  chosen with validation labels;
- price and weight medians and MADs in `add_anom` are computed on all rows of each frame, including the
  validation fold and the test set;
- categorical levels are built from train + test.

Its OOF score is therefore slightly optimistic. The code is the team's original version, unchanged.

In [1]:
"""Improved pipeline: pack-mismatch + strict category contradiction + phrase anomaly features."""
import argparse, json, hashlib, re, time, platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
WORD2NUM = {'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,
            'eight':8,'nine':9,'ten':10,'zero':0,'single':1}


def seller_folds(sellers, n_splits=5, seed=SEED):
    assignment = np.array([int(hashlib.sha256(f'{seed}:{s}'.encode('utf-8')).hexdigest()[:16], 16)
                           % n_splits for s in sellers.astype(str)])
    for fold in range(n_splits):
        fit, valid = np.flatnonzero(assignment != fold), np.flatnonzero(assignment == fold)
        if not len(fit) or not len(valid):
            raise ValueError('Empty seller fold')
        yield fit, valid


def extract_main_items(text):
    t = str(text).lower()
    t = re.sub(r'also includes \d+ pcs? of free accessory[^.]*\.?', ' ', t)
    t = re.sub(r'free accessory[^.]*\.?', ' ', t)
    pats = [
        r'included main items:\s*(\d+)',
        r'(\d+)\s+individually boxed main items',
        r'(one|two|three|four|five|six|seven|eight|nine|ten)\s+individually boxed main items',
        r'(\d+)\s+main items',
        r'(one|two|three|four|five|six|seven|eight|nine|ten)\s+main items',
        r'pack of (one|two|three|four|five|six|seven|eight|nine|ten) main items',
        r'pack of (\d+) main items',
        r'bundle quantity:\s*(\d+)',
        r'(\d+)\s*-?pack',
        r'sold as a (\d+)-pack',
        r'\(quantity\s*(\d+)\)',
        r'quantity\s*(\d+)',
        r'(\d+)\s*units',
        r'package contains\s*(\d+)',
        r'(\d+)\s*pcs',
        r'pack of (one|two|three|four|five|six|seven|eight|nine|ten)',
    ]
    for p in pats:
        m = re.search(p, t)
        if m:
            v = m.group(1)
            if v.isalpha():
                return WORD2NUM.get(v)
            try: return int(v)
            except: pass
    return None


def extract_free_accessory(text):
    m = re.search(r'also includes (\d+) pcs? of free accessory', str(text).lower())
    return int(m.group(1)) if m else 0


def extract_product_phrase(row):
    desc = str(row['description']) if pd.notna(row['description']) else ''
    m = re.search(r'([a-z][a-z\s\-/]{2,40}?)\s+by\s+(?:mira|noro|velto|tenso|aster)', desc.lower())
    if m: return m.group(1).strip()
    t = str(row['title']).lower() if pd.notna(row['title']) else ''
    m = re.search(r'([a-z][a-z\s\-/]{2,40}?)\s*[—\-/]', t)
    return m.group(1).strip() if m else ''


def features(df):
    x = df.copy()
    title = x.title.fillna('').astype(str).str.lower()
    desc  = x.description.fillna('').astype(str).str.lower()
    x['text'] = (title + ' ' + desc).str.replace(r'\s+', ' ', regex=True).str.strip()
    x['text_is_short'] = (x['text'].str.len() < 20).astype(int)

    x['main_items'] = x['text'].apply(extract_main_items)
    x['free_acc']   = x['text'].apply(extract_free_accessory)
    pk, mi = x.pack_size, x.main_items
    x['main_vs_pack'] = np.where(pk.isna() | mi.isna(), -1,
                          np.where(pk == mi, 1, 0)).astype(int)
    x['mvp_mismatch']    = (x.main_vs_pack == 0).astype(int)
    x['pack_vs_main']    = pk - mi
    x['pack_vs_main_abs']= (pk - mi).abs()
    x['pack_missing']    = pk.isna().astype(int)
    x['main_missing']    = mi.isna().astype(int)
    x['total_items_text']= mi.fillna(0) + x['free_acc'].fillna(0)
    x['total_vs_pack']   = x['total_items_text'] - pk

    x['product_phrase'] = (x.apply(extract_product_phrase, axis=1)
                             .str.replace(r'\s+', ' ', regex=True).str.strip())

    for c in ['title','description','declared_category','brand','condition']:
        x['isna_'+c] = x[c].isna().astype(int)
    x['isna_price']  = x['price_pln'].isna().astype(int)
    x['isna_weight'] = x['shipping_weight_kg'].isna().astype(int)

    for c in ['price_pln','shipping_weight_kg','pack_size']:
        x['log_'+c] = np.log1p(x[c].clip(lower=0))
    x['price_per_item']   = np.log1p(x.price_pln / x.pack_size.clip(lower=1))
    x['mass_per_item']    = np.log1p(x.shipping_weight_kg / x.pack_size.clip(lower=1))
    x['price_per_weight'] = np.log1p(x.price_pln / (x.shipping_weight_kg + 1e-3))

    x['title_len']  = title.str.len()
    x['desc_len']   = desc.str.len()
    x['title_word_ct'] = title.str.split().str.len().fillna(0)
    x['desc_word_ct']  = desc.str.split().str.len().fillna(0)
    x['desc_title_ratio'] = x['desc_len'] / (x['title_len'] + 1)

    cat = x.declared_category.fillna('missing').astype(str).str.lower()
    x['category_in_text'] = [int(c in t) for c, t in zip(cat, x['text'])]
    return x.replace({pd.NA: np.nan})


def add_phrase_features(x, xt, fit_mask):
    def phrase_stats_agg(s):
        if len(s.dropna()) < 5: return (0.0, None)
        vc = s.value_counts(normalize=True)
        return (float(vc.iloc[0]), vc.index[0])

    fit_x = x.iloc[fit_mask]
    stats = fit_x.groupby('product_phrase')['declared_category'].agg(phrase_stats_agg)
    purity_map = {p: stats[p][0] for p in stats.index}
    cat_map    = {p: stats[p][1] for p in stats.index}
    pack_mode_map = fit_x.groupby('product_phrase')['pack_size'].agg(
        lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) > 0 else None).to_dict()

    for df in (x, xt):
        df['phrase_purity']    = df['product_phrase'].map(purity_map).fillna(0.0)
        df['phrase_modal_cat'] = df['product_phrase'].map(cat_map)
        df['cat_contradicts_strict'] = (
            (df.phrase_purity >= 0.90) & df.phrase_modal_cat.notna() &
            df.declared_category.notna() &
            (df.phrase_modal_cat != df.declared_category)
        ).astype(int)
        df['pack_mode_pp'] = df['product_phrase'].map(pack_mode_map)
        df['pack_anomaly_pp'] = (
            df.pack_mode_pp.notna() & df.pack_size.notna() &
            (df.pack_size != df.pack_mode_pp)
        ).astype(int)

    phrase_counts = fit_x['product_phrase'].value_counts()
    trusted = set(phrase_counts[phrase_counts >= 8].index)

    grp_key = lambda df: df['product_phrase'].astype(str) + '||' + df['declared_category'].astype(str)
    fit_grp = fit_x.assign(_grp=grp_key(fit_x))
    grp_counts = fit_grp['_grp'].value_counts()
    trusted_pc = set(grp_counts[grp_counts >= 6].index)

    for df in (x, xt):
        df['_grp_pc'] = grp_key(df)

    def add_anom(df):
        for feat_name, f in [
            ('price',       df['price_pln']),
            ('weight',      df['shipping_weight_kg']),
            ('price_item',  df['price_pln'] / df['pack_size'].clip(lower=1)),
            ('weight_item', df['shipping_weight_kg'] / df['pack_size'].clip(lower=1)),
        ]:
            tmp = df.assign(_v=f.values)

            tmp['_med_pp'] = tmp.groupby('product_phrase', dropna=False)['_v'].transform('median')
            tmp['_abs_pp'] = (tmp['_v'] - tmp['_med_pp']).abs()
            tmp['_mad_pp'] = (tmp.groupby('product_phrase', dropna=False)['_abs_pp']
                                .transform('median') + 1e-6)
            z_pp = (tmp['_v'] - tmp['_med_pp']) / tmp['_mad_pp']

            df[f'{feat_name}_z_pp2']       = z_pp.values
            df[f'{feat_name}_hi_pp2']      = ((z_pp.abs() > 4) & df['product_phrase'].isin(trusted)).astype(int).values
            df[f'{feat_name}_extreme_pp2'] = ((z_pp.abs() > 6) & df['product_phrase'].isin(trusted)).astype(int).values

            tmp['_med_pc'] = tmp.groupby('_grp_pc', dropna=False)['_v'].transform('median')
            tmp['_abs_pc'] = (tmp['_v'] - tmp['_med_pc']).abs()
            tmp['_mad_pc'] = (tmp.groupby('_grp_pc', dropna=False)['_abs_pc']
                                .transform('median') + 1e-6)
            z_pc = (tmp['_v'] - tmp['_med_pc']) / tmp['_mad_pc']

            df[f'{feat_name}_z_pc']       = z_pc.values
            df[f'{feat_name}_extreme_pc'] = ((z_pc.abs() > 6) & df['_grp_pc'].isin(trusted_pc)).astype(int).values

    add_anom(x)
    add_anom(xt)

    for df in (x, xt):
        df['any_extreme_anom'] = (
            df['price_item_extreme_pp2'].astype(bool) |
            df['weight_item_extreme_pp2'].astype(bool) |
            df['price_item_extreme_pc'].astype(bool) |
            df['weight_item_extreme_pc'].astype(bool)
        ).astype(int)
        df['anom_pp_minus_pc']    = (df['price_item_z_pp2'].abs() - df['price_item_z_pc'].abs()).fillna(0)
        df['anom_wt_pp_minus_pc'] = (df['weight_item_z_pp2'].abs() - df['weight_item_z_pc'].abs()).fillna(0)
        df['extreme_AND_mismatch']= (df['any_extreme_anom'] * df['mvp_mismatch']).astype(int)
        df['rule_score'] = (
            df['mvp_mismatch']*10.0
            + df['cat_contradicts_strict']*5.0
            + df['any_extreme_anom']*2.0
            + df['price_item_hi_pp2']*1.0
            + df['weight_item_hi_pp2']*1.0
        )
    return x, xt


FEATS = [
    'mvp_mismatch','cat_contradicts_strict','pack_anomaly_pp',
    'price_item_extreme_pp2','weight_item_extreme_pp2',
    'price_item_extreme_pc','weight_item_extreme_pc',
    'any_extreme_anom','extreme_AND_mismatch',
    'anom_pp_minus_pc','anom_wt_pp_minus_pc',
    'price_item_hi_pp2','weight_item_hi_pp2',
    'rule_score',
    'price_item_z_pp2','weight_item_z_pp2','price_z_pp2','weight_z_pp2',
    'price_item_z_pc','weight_item_z_pc','price_z_pc','weight_z_pc',
    'pack_vs_main','pack_vs_main_abs','main_items','free_acc',
    'log_price_pln','log_shipping_weight_kg','log_pack_size',
    'price_per_item','mass_per_item','price_per_weight',
    'title_len','desc_len','desc_title_ratio',
    'isna_title','isna_description','isna_declared_category',
    'isna_brand','isna_condition','isna_price','isna_weight',
    'category_in_text','text_is_short','phrase_purity',
]
CATS = ['declared_category','brand','condition']


def resolve_data_dir():
    local = Path('data/public')
    if (local / 'train.csv').exists():
        return local
    candidates = sorted(Path('/kaggle/input').rglob('train.csv'))
    candidates = [p.parent for p in candidates if (p.parent / 'sample_submission.csv').exists()]
    if len(candidates) != 1:
        raise ValueError('Specify --data-dir: could not identify one competition dataset')
    return candidates[0]


def run(data_dir, output_dir='baseline_output'):
    started = time.perf_counter()
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(data_dir / 'train.csv')
    test  = pd.read_csv(data_dir / 'test.csv')
    sample= pd.read_csv(data_dir / 'sample_submission.csv')

    assert train.target.isin([0, 1]).all()
    assert set(train.seller_id).isdisjoint(test.seller_id)
    assert train.id.is_unique and test.id.is_unique
    assert set(train.id).isdisjoint(test.id)
    assert sample.id.tolist() == test.id.tolist()

    print('Shapes:', train.shape, test.shape)
    print('Target proportions:', train.target.value_counts(normalize=True).to_dict())

    x  = features(train.drop(columns='target'))
    xt = features(test)
    y  = train.target.to_numpy()

    oof = np.zeros(len(train))
    predictions = np.zeros(len(test))
    scores = []
    fold_assignment = np.zeros(len(train), dtype=int)

    for fold, (fit, valid) in enumerate(seller_folds(train.seller_id), 1):
        assert set(train.iloc[fit].seller_id).isdisjoint(train.iloc[valid].seller_id)
        fold_assignment[valid] = fold

        x_fold = x.copy()
        xt_fold = xt.copy()
        x_fold, xt_fold = add_phrase_features(x_fold, xt_fold, fit_mask=fit)

        for c in CATS:
            cats = pd.Categorical(pd.concat([x_fold[c], xt_fold[c]]).fillna('missing')).categories
            x_fold[c]  = pd.Categorical(x_fold[c].fillna('missing'),  categories=cats)
            xt_fold[c] = pd.Categorical(xt_fold[c].fillna('missing'), categories=cats)

        ALL = FEATS + CATS
        X  = x_fold[ALL].copy()
        Xt = xt_fold[ALL].copy()
        for c in FEATS:
            X[c]  = pd.to_numeric(X[c],  errors='coerce').astype('float32')
            Xt[c] = pd.to_numeric(Xt[c], errors='coerce').astype('float32')

        m = lgb.LGBMClassifier(
            n_estimators=1500, learning_rate=0.03, num_leaves=95,
            min_child_samples=20, colsample_bytree=0.85, subsample=0.85,
            reg_lambda=1.0, objective='binary', metric='auc',
            verbosity=-1, n_jobs=-1, random_state=SEED + fold,
        )
        m.fit(X.iloc[fit], y[fit],
              eval_set=[(X.iloc[valid], y[valid])],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
              categorical_feature=CATS)

        oof[valid] = m.predict_proba(X.iloc[valid])[:, 1]
        predictions += m.predict_proba(Xt)[:, 1] / 5
        score = roc_auc_score(y[valid], oof[valid])
        scores.append(float(score))
        print(f'Fold {fold}: ROC-AUC={score:.6f}  best_iter={m.best_iteration_}')

    report = {'fold_roc_auc': scores, 'mean_roc_auc': float(np.mean(scores)),
              'std_roc_auc': float(np.std(scores)), 'oof_roc_auc': float(roc_auc_score(y, oof)),
              'oof_average_precision': float(average_precision_score(y, oof)),
              'elapsed_seconds': time.perf_counter() - started,
              'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                           'pandas': pd.__version__, 'sklearn': sklearn.__version__},
              'seed': SEED,
              'fold_policy': 'SHA256(seed:seller_id) first 16 hex digits modulo 5',
              'data_sha256': {name: hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
                              for name in ['train.csv', 'test.csv', 'sample_submission.csv']},
              'fold_assignment_sha256': hashlib.sha256(
                  pd.DataFrame({'id': train.id, 'fold': fold_assignment}).to_csv(index=False, lineterminator='\n').encode('utf-8')
              ).hexdigest()}
    sample['target'] = predictions
    assert np.isfinite(predictions).all() and ((predictions >= 0) & (predictions <= 1)).all()
    sample.to_csv(output_dir / 'submission.csv', index=False)
    pd.DataFrame({'id': train.id, 'target': y, 'prediction': oof,
                  'fold': fold_assignment}).to_csv(output_dir / 'oof.csv', index=False)
    (output_dir / 'cv_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report

## Validation and submission

In [2]:
data_dir = resolve_data_dir()
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('baseline_output')
report = run(data_dir, output_dir)
print('lightgbm', lgb.__version__)
print('Submission:', output_dir / 'submission.csv')

Shapes: (16000, 11) (10000, 10)
Target proportions: {0: 0.8299375, 1: 0.1700625}


Fold 1: ROC-AUC=0.968851  best_iter=406


Fold 2: ROC-AUC=0.965451  best_iter=210


Fold 3: ROC-AUC=0.975819  best_iter=425


Fold 4: ROC-AUC=0.971604  best_iter=422


Fold 5: ROC-AUC=0.972435  best_iter=730
{
  "fold_roc_auc": [
    0.9688511011790644,
    0.9654511808968589,
    0.975818580119557,
    0.9716038412220873,
    0.9724350578821604
  ],
  "mean_roc_auc": 0.9708319522599457,
  "std_roc_auc": 0.00349023463631167,
  "oof_roc_auc": 0.9683037761457876,
  "oof_average_precision": 0.9277677473119099,
  "elapsed_seconds": 24.710269474999997,
  "versions": {
    "python": "3.13.17",
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "sklearn": "1.6.1"
  },
  "seed": 42,
  "fold_policy": "SHA256(seed:seller_id) first 16 hex digits modulo 5",
  "data_sha256": {
    "train.csv": "aefc970e5efc4435cecfe76b901149fdd9ffd06bc62a3f62773379902b34195c",
    "test.csv": "10423bf383cfd0bce44dde6459f48fcfee41a6e7ffbc60f8ba4c01de4fdaae13",
    "sample_submission.csv": "989f9e1ffdbce58f093f1076b85c988fdbca2dbed35534a004b7944afa5f930a"
  },
  "fold_assignment_sha256": "dda65b9df2a806620c958e7c38199e2b9ceb1b0190992b8316f40bdf93f36b8c"
}
lightgbm 4.6.0
Submission: b